# 검수 확인서 테스트

In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain_teddynote import logging
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents import create_agent
from pathlib import Path
from IPython.display import Markdown, display

from dotenv import load_dotenv

load_dotenv(override=True)

# 추적을 위한 프로젝트 이름 설정
logging.langsmith("Samsung-Asset-AI-Portal")

# prompt 모듈에서 필요한 항목 import
from prompt import (
    _SYSTEM_PROMPT,
    _SYSTEM_PROMPT_ENG,
    get_prompt_pdf_text_to_markdown,
    get_prompt_text_to_markdown,
    get_prompt_pdf_text_to_markdown_validate,
    get_prompt_text_to_markdown_validate,
    get_prompt_confirmed_expected_clarification,
    get_prompt_text_to_markdown_validate_report,
    get_prompt_text_to_markdown_error_fix,
    get_prompt_confirmed_expected_clarification_validate_report,
    get_prompt_confirmed_expected_clarification_error_fix
)

# document_parser 모듈에서 필요한 함수들 import
from document_parser import (
    extract_pdf_with_docling,
    extract_pdf_with_pdfplumber,
    parser_excel,
    get_file_path,
    get_last_ai_message
)

from utils import format_messages

_original_text = None

# LLM 모델 생성

In [ ]:
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")

def create_llm_model():

    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=0.0,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.1로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    # llm = init_chat_model(
    #     "openai:gpt-4o",
    #     temperature=0.0,
    #     top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
    # )
    return llm

# excel to markdown

In [ ]:
# 싱글톤 
def text_to_markdown_with_llm(document_text: str):
    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown(document_text)
    print("########## text to markdown prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def text_to_markdown_with_plan(document_text: str):
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown(document_text)
    print("########## text to markdown prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

def convert_excel_to_markdown(file_path: str):
    # 엑셀 파일 경로 확인
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    # 엑셀 파일에서 텍스트 추출
    _original_text = parser_excel(file_path)

    # 텍스트를 markdown으로 변환
    # response_markdown = text_to_markdown_with_plan(document_text_excel)
    response_markdown = text_to_markdown_with_llm(_original_text)
    # markdown_text = get_last_ai_message(response_markdown)

    return response_markdown


# document to markdown

In [ ]:
def document_to_markdown(file_path: str):
    path_obj = Path(file_path)
    file_name = path_obj.name  # 파일명 (확장자 포함)
    file_ext = path_obj.suffix  # 확장자 (점 포함, 예: .pdf)
    markdown_text = None

    if file_ext == ".xlsx" or file_ext == ".xls":
        markdown_text = convert_excel_to_markdown(file_path)
    elif file_ext == ".pdf":
        print("pdf - 공사중...")
    else:
        print("지원하지 않는 파일 형식입니다.")

    return markdown_text


# markdown 정리 문서 검수 보고서 작성

In [ ]:
# 싱글톤 
def text_to_markdown_validate_report_with_llm(markdown_text: str, original_text: str):
    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown_validate_report(markdown_text, original_text)
    print("########## 검수보고서 작성 prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def text_to_markdown_validate_report_with_plan(markdown_text: str, original_text: str):
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown_validate_report(markdown_text, original_text)
    print("########## 검수보고서 작성 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# markdown 정리 문서 오류 수정

In [ ]:
# 싱글톤 
def text_to_markdown_error_fix_with_llm(markdown_text: str, validate_report: str, original_text: str):
    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_text_to_markdown_error_fix(markdown_text, original_text, validate_report)
    print("########## 오류 수정 prompt ##########")
    print(human_prompt)
    human_msg = HumanMessage(human_prompt)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def text_to_markdown_error_fix_with_plan(markdown_text: str, validate_report: str, original_text: str):
    llm = create_llm_model()
    human_prompt = get_prompt_text_to_markdown_error_fix(markdown_text, original_text, validate_report)
    print("########## 오류 수정 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# 확정분/청구분 정리

In [ ]:
# markdown으로 정리된 지시서를 받아 확정분과 청구분으로 구분하여 데이터 정리
# 싱글톤
def classify_confirmed_expected_from_instruction_with_llm(instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_confirmed_expected_clarification(instruction_markdown)
    human_msg = HumanMessage(human_prompt)
    print("########## 확정분/청구분 구분 prompt ##########")
    print(human_prompt)
    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def classify_confirmed_expected_from_instruction_with_plan(instruction_markdown: str):
    llm = create_llm_model()
    human_prompt = get_prompt_confirmed_expected_clarification(instruction_markdown)
    print("########## 확정분/청구분 구분 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# 확정분/청구분 분류 결과 검수

In [ ]:
# 싱글톤
def classify_confirmed_expected_validate_report_with_llm(clarification_markdown: str, instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_confirmed_expected_clarification_validate_report(clarification_markdown, instruction_markdown)
    human_msg = HumanMessage(human_prompt)
    print("########## 확정분/청구분 검수 prompt ##########")
    print(human_prompt)
    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def classify_confirmed_expected_validate_report_with_plan(clarification_markdown: str, instruction_markdown: str):
    llm = create_llm_model()
    human_prompt = get_prompt_confirmed_expected_clarification_validate_report(clarification_markdown, instruction_markdown)
    print("########## 확정분/청구분 검수 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# 확정분/청구분 오류 수정

In [ ]:
# 싱글톤
def classify_confirmed_expected_error_fix_with_llm(validate_report: str, clarification_markdown: str, instruction_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_prompt = get_prompt_confirmed_expected_clarification_error_fix(validate_report, clarification_markdown, instruction_markdown)
    human_msg = HumanMessage(human_prompt)
    print("########## 확정분/청구분 오류 수정 prompt ##########")
    print(human_prompt)
    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response


# todos 미들웨어 사용
def classify_confirmed_expected_error_fix_with_plan(validate_report: str, clarification_markdown: str, instruction_markdown: str):
    llm = create_llm_model()
    human_prompt = get_prompt_confirmed_expected_clarification_error_fix(validate_report, clarification_markdown, instruction_markdown)
    print("########## 확정분/청구분 오류 수정 prompt ##########")
    print(human_prompt)
    todo_agent = create_agent(
        llm,
        system_prompt=_SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()]
    )

    response = todo_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": human_prompt
                }
            ],
        }
    )

    return response

# TEST

In [ ]:
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/ABL_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/DB_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프(액티브)_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
# _password = '345678'



### markdown 변환

In [ ]:
# result_markdown = document_to_markdown(_document_file_path)

_original_text = parser_excel(_document_file_path)
result_markdown = text_to_markdown_with_llm(_original_text)

In [ ]:
print(f"file_name : {_document_file_path}")
display(Markdown(result_markdown.content))

In [ ]:
# format_messages(result_markdown["messages"])

In [ ]:
# print(f"file_name : {_document_file_path}")
# last_ai_message = get_last_ai_message(result_markdown)
# print(last_ai_message)
# display(Markdown(last_ai_message.content))

### 검수

In [ ]:
# print("####### original_text #######")
# print(_original_text)

In [ ]:
validate_report = text_to_markdown_validate_report_with_llm(result_markdown.content, _original_text)
# print(validate_report)

In [ ]:
# validate_report = get_last_ai_message(validate_report)
# display(Markdown(validate_report.content))

In [ ]:
display(Markdown(validate_report.content))

### 검수 결과 오류 수정

In [ ]:
fixed_markdown = text_to_markdown_error_fix_with_llm(result_markdown.content, validate_report.content, _original_text)
# print(fixed_markdown)

In [ ]:
display(Markdown(fixed_markdown.content))

### 재 검수

In [ ]:
validate_report = text_to_markdown_validate_report_with_llm(fixed_markdown.content, _original_text)
# print(validate_report)

In [ ]:
display(Markdown(validate_report.content))

### 검수 결과 오류 재 수정

In [ ]:
fixed_markdown = text_to_markdown_error_fix_with_llm(fixed_markdown.content, validate_report.content, _original_text)
# print(fixed_markdown)

In [ ]:
display(Markdown(fixed_markdown.content))

### 확정분/청구분 분류

In [ ]:
classify_confirmed_expected_data = classify_confirmed_expected_from_instruction_with_llm(fixed_markdown.content)
# print(fixed_markdown)

In [ ]:
display(Markdown(classify_confirmed_expected_data.content))

### 확정분/청구분 검수

In [ ]:
classify_confirmed_expected_validate_report = classify_confirmed_expected_validate_report_with_llm(
    classify_confirmed_expected_data.content, 
    fixed_markdown.content
)
# print(fixed_markdown)

In [ ]:
display(Markdown(classify_confirmed_expected_validate_report.content))

### 확정분/청구분 오류 수정

In [ ]:
classify_confirmed_expected_fix_data = classify_confirmed_expected_error_fix_with_llm(
    classify_confirmed_expected_validate_report.content,
    classify_confirmed_expected_data.content, 
    fixed_markdown.content
)
# print(fixed_markdown)

In [ ]:
display(Markdown(classify_confirmed_expected_fix_data.content))